In [24]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [25]:
journey_att=[]

journery_embedd=inputs[1]
for el in inputs:
  journey_att.append(torch.dot(journery_embedd,el))



journey_att


normalised_journey_att=torch.tensor(journey_att)/torch.sum(torch.tensor(journey_att))
normalised_journey_att

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [26]:
# lets use softmax instead of normalise because softmax give lower value to lower one and higher to higher one and thats what we want


softmax_journey_att=torch.softmax(torch.tensor(journey_att),dim=0)
softmax_journey_att

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

Here’s your same content, just cleaned and formatted in simple language 👇

---

## Softmax in PyTorch

also instead of using **vanila softmax** or say **naive softmax**, pytorch uses formula:

[
e^{(x - max(x))} ; / ; \sum \left(e^{(x - max(x))}\right)
]

for preventing **overflow** (DSA approach)

---


In [27]:

from copy import  deepcopy
class Attention:
  def __init__(self,embeded_vec:torch.tensor):
    self.embeded_vec=deepcopy(embeded_vec)


  def initiate(self):
    weight=self.embeded_vec @ self.embeded_vec.T
    print(f"weights shape {self.embeded_vec.shape} dim {self.embeded_vec.dim()}")


    # normalising relative vector

    normalised_relative_vec=torch.softmax(weight,dim=0)
    print(f"weight shape {weight.shape} dim {weight.dim()}")


    relative_vec=normalised_relative_vec @ self.embeded_vec
    print(f"relative_vec shape {relative_vec.shape} dim {relative_vec.dim()}")


    return relative_vec







In [28]:
attention=Attention(embeded_vec=torch.tensor(inputs))

/tmp/ipykernel_15216/4038510332.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention=Attention(embeded_vec=torch.tensor(inputs))


In [29]:
attention.initiate()

weights shape torch.Size([6, 3]) dim 2
weight shape torch.Size([6, 6]) dim 2
relative_vec shape torch.Size([6, 3]) dim 2


tensor([[0.4017, 0.5023, 0.5059],
        [0.5595, 0.7824, 0.6953],
        [0.5538, 0.7686, 0.6834],
        [0.3369, 0.4647, 0.4119],
        [0.3525, 0.4059, 0.3657],
        [0.3856, 0.5761, 0.5077]])

In [30]:
import torch.nn as nn


In [39]:
class SelfAttention(nn.Module):
  def __init__(self,input_dim,output_dim):
    super().__init__()
    self.Q=nn.Parameter(torch.rand(input_dim,output_dim))
    self.K=nn.Parameter(torch.rand(input_dim,output_dim))
    self.V=nn.Parameter(torch.rand(input_dim,output_dim))

  def forward(self,x):
    query=x @ self.Q
    keys=x @ self.K
    values=x @ self.V

    weights=query @ keys.transpose(-2, -1)

    weights_normalised=torch.softmax(
        (weights/keys.shape[-1]**0.5),dim=-1
    )

    embedds=weights_normalised @ values

    return embedds

In [50]:
torch.manual_seed(123)
self_attention=SelfAttention(input_dim=3,output_dim=3)

In [51]:
self_attention(inputs)

tensor([[0.6692, 1.0276, 1.1106],
        [0.6864, 1.0577, 1.1389],
        [0.6860, 1.0570, 1.1383],
        [0.6738, 1.0361, 1.1180],
        [0.6711, 1.0307, 1.1139],
        [0.6783, 1.0441, 1.1252]], grad_fn=<MmBackward0>)

# for better weight initialisation use Linear instead of parameter

In [64]:
class SelfAttention_v2(nn.Module):
  def __init__(self,input_dim,output_dim,bias=False):
    super().__init__()
    self.Q=nn.Linear(input_dim,output_dim,bias=bias)
    self.K=nn.Linear(input_dim,output_dim,bias=bias)
    self.V=nn.Linear(input_dim,output_dim,bias=bias)

  def forward(self,x):
    query=self.Q(x)
    keys=self.K(x)
    values=self.V(x)

    weights=query @ keys.transpose(-2, -1)

    weights_normalised=torch.softmax(
        (weights/keys.shape[-1]**0.5),dim=-1
    )

    embedds=weights_normalised @ values

    return embedds

In [65]:
torch.manual_seed(123)
self_attention=SelfAttention_v2(input_dim=3,output_dim=3)

In [66]:
self_attention(inputs)

tensor([[ 0.2633,  0.4277, -0.1353],
        [ 0.2641,  0.4296, -0.1350],
        [ 0.2641,  0.4296, -0.1350],
        [ 0.2647,  0.4316, -0.1381],
        [ 0.2642,  0.4303, -0.1373],
        [ 0.2648,  0.4316, -0.1375]], grad_fn=<MmBackward0>)